The notebook uses a local language model (Gemma 3 1B) and uses RAG to retrieve information from Wikipedia pages about two niche topics and tests whether the llm is able to answer questions about the topics given knowledge from the vector Data Base

In [24]:
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter



In [25]:
# List of URLs to load documents from
urls = [
    "https://en.wikipedia.org/wiki/Philadelphia_Experiment/",
    "https://en.wikipedia.org/wiki/24_Hours_of_Lemons/",

]
# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

In [26]:
# investigate the docs and docs_list objects
print(type(docs))
print(type(docs_list))

<class 'list'>
<class 'list'>


In [27]:
print(len(docs), len(docs_list))

2 2


In [28]:
# split the retrived document into chunks. RecursiveCharacterTextSplitter is a text splitter that recursively splits text into smaller chunks based on a set of rules.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)
doc_split = text_splitter.split_documents(docs_list)

In [29]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [31]:
# create vector embeddings for the tokens in the documents
# openAI embeddings are used to create vector embedding
# SklearnVectorStore is used to store the vector embeddings
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_openai import OpenAIEmbeddings

vector_store = SKLearnVectorStore.from_documents(documents = doc_split, 
                                    embedding=embeddings,)
retriever = vector_store.as_retriever(k=2) # topk = 2


In [32]:
# set up the LLM and prompt template
from langchain_ollama import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [33]:
prompt = PromptTemplate(
    template = """You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,input_variables=["question", "documents"]
)

In [34]:
llm = ChatOllama(model="gemma3:1b", temperature=0.0)

In [35]:
# chainign together the llm, prompt and string parser
rag_chain = prompt | llm |  StrOutputParser()

In [36]:
#final integration into a rag application
class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
    
    def run(self, question):
        documents = self.retriever.invoke(question) # get the top k documents
        # Extract content from retrieved documents
        doc_texts = "\n".join([doc.page_content for doc in documents])
        #Get answer from llm
        answer = self.rag_chain.invoke({"question":question, "documents":doc_texts})
        return answer


In [37]:
# Initialize the RAG application
rag_application = RAGApplication(retriever, rag_chain)
# Example usage
question = "What is the Philadelphia Experiment?"
answer = rag_application.run(question)
print("Question:", question)
print("Answer:", answer)

Question: What is the Philadelphia Experiment?
Answer: The Philadelphia Experiment was a project initiated in 1972 by the University of Pennsylvania's School of Engineering to investigate the effectiveness of a centralized, automated system for managing a large, complex infrastructure. It aimed to improve the efficiency and coordination of city services.
